# Simple GNN Test with 5 Simple Graphs

Further proof of concept for the GNN approach using 5 simple graphs.

In [ ]:
%load_ext autoreload
%autoreload 2

## Load data

**The 5 graph data have been saved. The following sections can be skipped!**

In [ ]:
# from image_processing_tools.util.load_files import find_files_by_pattern
# from image_processing_tools.image_class.image_container import ImageContainer
# from pathlib import Path
# import matplotlib
# # matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

# search_path = ("~/A8/PyG Data/simple-network-crops/",)
# file_pattern = ("CROP_*.tif",
#                 "MAX_*",
#                 "SHARPEST*",
#                 "MAX_CET145*")

# config = {
#     "preprocessing": {
#         "resize_image": False,
#         "max_dim": 1080,
#         "outlier_percentile": 0.35,
#         "quantization": "16bit"
#     }
# }

# # Find the files. The 'files' variable will hold the list of Path objects.
# files = find_files_by_pattern(search_path[0], file_pattern[0], verbose=True)

In [ ]:
# imgs = [ImageContainer(img,config) for img in files]
# imgs_merged = [img.merge() for img in imgs]

In [ ]:
# from image_processing_tools.dapi_tracing.connectivity_score import detect_nuclei_rf
# from image_processing_tools.rf_nuclei.rf_load_models import load_model
# from pathlib import Path

# model = load_model(Path('~/A8/PyG Data/simple-network-crops/batch_rf_model.joblib').expanduser())

# seg_output = []
# for i, img in enumerate(imgs_merged):
#     print(i)
#     tmp = detect_nuclei_rf(seg_image=img,model = model,nuclei_class=1)
#     seg_output.append(tmp)

## Build graph

In [ ]:
# from image_processing_tools.dapi_tracing.connectivity_score import filter_nuclei, extract_graph

# def extract_graph_pipeline_rf(seg_img,model,lower=0.01,upper=5):
#     labels, binary_mask_filled = detect_nuclei_rf(seg_img,model)
#     valid_nuclei_data,_,_ = filter_nuclei(labels,lower,upper)
#     nuclei_df, nuclei_centroids, paths_df, edge_index = extract_graph(valid_nuclei_data, seg_img, binary_mask_filled)
    
#     return nuclei_df, nuclei_centroids, paths_df, edge_index

# nuclei_df_list, paths_df_list, edge_index_list, centroid_list = [],[],[],[]

# for i, img in enumerate(imgs_merged):
#     seg_img = img
#     lower = 0.3 if i == 4 else 0.01
#     nuclei_df, nuclei_centroids, paths_df, edge_index = extract_graph_pipeline_rf(seg_img,model,lower)
#     nuclei_df_list.append(nuclei_df)
#     centroid_list.append(nuclei_centroids)
#     paths_df_list.append(paths_df)
#     edge_index_list.append(edge_index)

In [ ]:
# import pandas as pd

# pd.DataFrame(edge_index_list[1])

In [ ]:
# edge_labels_list = []
# edge_labels_list.append([1,0,1]) # T/F = 2/1
# edge_labels_list.append([0,0,1,1,0,1]) # 1/1
# edge_labels_list.append([1,0,1]) # 2/1
# edge_labels_list.append([1,0,0,0,1,0,0,0,1,0]) # 3/7
# edge_labels_list.append([1,0,0,0,0,1,0,0,0,0,1,0,0,0,0]) # 3/12

In [ ]:
# from image_processing_tools.dapi_tracing.gnn_data import create_pyg_data, save_pyg_dataset, load_pyg_dataset

# pyg_data_list = create_pyg_data(edge_indices = edge_index_list, 
#                                 nuclei_features_list = nuclei_df_list, 
#                                 path_features_list = paths_df_list, 
#                                 edge_labels_list = edge_labels_list,
#                                 images_list = imgs_merged,
#                                 centroids_list=centroid_list)

# save_pyg_dataset(pyg_data_list,'/home/wangchuyao/Projects/C_Albicans Thesis Project/7. Data/graph_datasets/5graphs_dapi')

from image_processing_tools.dapi_tracing.gnn_data import load_pyg_dataset
pyg_data_list = load_pyg_dataset('/home/wangchuyao/Projects/C_Albicans Thesis Project/7. Data/graph_datasets/5graphs_dapi')

## Overfit on all data

In [ ]:
# from image_processing_tools.dapi_tracing.gnn_data import train_overfit_test

# model_params = {'hidden_channels': 32,
#                 'dropout_p': 0}

# results = train_overfit_test(dataset=pyg_data_list,
#                              max_epochs=300,
#                              batch_size=4,
#                              learning_rate=0.1,
#                              model_params=model_params,
#                              patience=100,
#                              experiment='5_graphs_muon')

## Run 5 fold cv

In [ ]:
from image_processing_tools.dapi_tracing.gnn_train import n_fold_validation

model_params = {'hidden_channels': 64,
                'dropout_p': 0.5}

for i in range(3):
    print(f'Starting repeated CV {i}')
    results = n_fold_validation(dataset = pyg_data_list,
                                num_folds = 5,
                                max_epochs = 500,
                                batch_size = 4,
                                learning_rate = 0.1,
                                model_params=model_params,
                                patience=100,
                                degree_penalty_weight=5.0,
                                neg_sample_ratio=1.0,
                                experiment=f'5graphs_muon_topKDegreePenaltyRest0Diff_averageProb_negSampleLossEarlyStopAUC_residualConnectionNorm_angleNorm_dynamicEdgeWeight_64_normdPathLength_inGraphMeanScaling/repeat_{i}')

## Testing acyclic network

In [ ]:
# from image_processing_tools.dapi_tracing.train_sinkhorn import n_fold_validation_acyclic

# model_params = {'hidden_channels': 64,
#                 'dropout_p': 0.5}

# for i in range(3):
#     print(f'Starting repeated CV {i}')
#     results = n_fold_validation_acyclic(dataset = pyg_data_list,
#                                 num_folds = 5,
#                                 max_epochs = 500,
#                                 batch_size = 4,
#                                 learning_rate = 0.1,
#                                 model_params=model_params,
#                                 patience=100,
#                                 degree_penalty_weight=2.0,
#                                 neg_sample_ratio=1.0,
#                                 experiment=f'acyclic_topologicalGradient_topKDegreePenaltyRest0Diff_normdPathLength/repeat_{i}')